## 메모리 (Memory)
- **상태를 저장하고 복원하는 핵심 메커니즘**
- 대화의 맥락을 유지하고, 오류 발생 시 안전하게 복구, 사용자별로 개인화된 경험을 제공

<br>

### 2단계 메모리 시스템
- `LangGraph`는 단기 메모리와 장기 메모리를 분리하여 2단계로 관리

<table>
<thead>
<tr>
<th style="text-align: left;">메모리 유형</th>
<th style="text-align: left;">범위</th>
<th style="text-align: left;">구현</th>
<th style="text-align: left;">특징</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><strong>단기 메모리</strong></td>
<td style="text-align: left;">Thread-scoped (단일 대화)</td>
<td style="text-align: left;"><code>Checkpointer</code></td>
<td style="text-align: left;">대화 히스토리 유지, 세션 종료 시 정리 가능</td>
</tr>
<tr>
<td style="text-align: left;"><strong>장기 메모리</strong></td>
<td style="text-align: left;">User/App-scoped (스레드 간 공유)</td>
<td style="text-align: left;"><code>Store</code></td>
<td style="text-align: left;">사용자 프로필/선호도, 영구 저장, 시맨틱 검색 가능</td>
</tr>
</tbody>
</table>

#### 단기 메모리 (`Checkpointer`)
- **`Checkpointer`는 게임의 세이브 포인트와 같은 것.**
- **그래프 실행의 특정 시점에서 상태를 저장하여 나중에 그 지점부터 다시 시작**

<br>

<table>
<thead>
<tr>
<th style="text-align: left;">기능</th>
<th style="text-align: left;">설명</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><strong>대화 연속성</strong></td>
<td style="text-align: left;">동일한 <code>thread_id</code>로 이전 대화 맥락 유지</td>
</tr>
<tr>
<td style="text-align: left;"><strong>오류 복구</strong></td>
<td style="text-align: left;">실행 중 문제 발생 시 마지막 안정 상태로 복원</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Human-in-the-Loop</strong></td>
<td style="text-align: left;">특정 지점에서 인간의 개입과 승인 허용</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Time Travel</strong></td>
<td style="text-align: left;">과거 체크포인트로 돌아가 다른 경로 탐색</td>
</tr>
</tbody>
</table>

<br>

#### 장기 메모리 (`Store`)
- **`Store`는 스레드 간 공유 가능한 Key-Value 저장소**
  - `Namespace`로 데이터를 구조화하며, `index` 설정으로 시맨틱 검색도 지원


```python
from langgraph.store.memory import InMemoryStore

store = InMemoryStore(
    index={
        "embed": embeddings,   # 임베딩 모델
        "dims": 1536,          # 임베딩 차원
        "fields": ["text"],    # 임베딩할 필드
    }
)

# Namespace 기반 저장
store.put(("user_123", "preferences"), "food", {"text": "매운 음식을 좋아합니다"})

# 시맨틱 검색
results = store.search(("user_123", "preferences"), query="음식 취향", limit=3)
```

<br>

<hr>

<br>

### 메모리 기능
- **메모리 기능이란 컴퓨터 프로그램이 이전에 일어난 일을 "기억"하는 기능입니다. `LangGraph`에서는 이를 체크포인터(`Checkpointer`)**

<br>

#### 메모리 사용의 핵심 장점

<table>
<thead>
<tr>
<th style="text-align: left;">장점</th>
<th style="text-align: left;">설명</th>
<th style="text-align: left;">예시</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><strong>자연스러운 대화</strong></td>
<td style="text-align: left;">이전 내용을 참고해서 대답</td>
<td style="text-align: left;">"아까 말한 그 책 제목이 뭐였죠?"</td>
</tr>
<tr>
<td style="text-align: left;"><strong>개인화</strong></td>
<td style="text-align: left;">사용자마다 다른 정보 기억</td>
<td style="text-align: left;">Alice와 Bob에게 각각 다른 응답</td>
</tr>
<tr>
<td style="text-align: left;"><strong>안전한 실행</strong></td>
<td style="text-align: left;">오류 발생 시 복구 가능</td>
<td style="text-align: left;">5단계에서 실패 → 5단계부터 재시작</td>
</tr>
</tbody>
</table>

<br>

### 메모리 기본 설정
- **`LangGraph`에서 메모리를 사용**
  - `thread_id` : 대화 세션을 구분하는 고유 식별자

<br>

```python
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver  # 체크포인터 import

# ... 그래프 정의 ...

memory = InMemorySaver()                              # 체크포인터 생성
app = graph.compile(checkpointer=memory)             #  그래프에 연결

# 실행 시 thread_id로 세션 구분
config = {"configurable": {"thread_id": "user_123"}}
app.invoke(initial_state, config=config)
```


<br>

#### 숫자 누적 계산 예제
- 숫자를 계속 더해가면서 결과를 기억하는 계산기

In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict

- 상태 정의

In [2]:
class CalculatorState(TypedDict):
    total: int
    history: list

- 노드 함수 정의

In [11]:
def add_number(state: CalculatorState) -> dict:
    """새 숫자를 더하고 기록을 남깁니다"""
    current_total = state["total"]
    number_to_add = 10  # 예제에서는 항상 10을 더함

    new_total = current_total + number_to_add
    new_history = state["history"] + [f"{current_total} + {number_to_add} = {new_total}"]

    print(f"계산: {current_total} + {number_to_add} = {new_total}")

    return {
        "total": new_total,
        "history": new_history
    }

- 그래프 구성

In [12]:
graph = StateGraph(CalculatorState)
graph.add_node("add", add_number)
graph.add_edge(START, "add")
graph.add_edge("add", END)

- 메모리 연결

In [13]:
memory = InMemorySaver()
app = graph.compile(checkpointer=memory)

- 실행

In [14]:
config = {"configurable": {"thread_id": "calculator_session"}}

In [15]:
print("=== 첫 번째 계산 ===")
result = app.invoke({"total": 0, "history": []}, config=config)
print(f"결과: total={result['total']}, history={result['history']}\n")

print("=== 두 번째 계산 ===")
result = app.invoke({}, config=config) 
print(f"결과: total={result['total']}, history={result['history']}\n")

print("=== 세 번째 계산 ===")
result = app.invoke({}, config=config)  
print(f"결과: total={result['total']}, history={result['history']}")

=== 첫 번째 계산 ===
계산: 0 + 10 = 10
결과: total=10, history=['0 + 10 = 10']

=== 두 번째 계산 ===
계산: 10 + 10 = 20
결과: total=20, history=['0 + 10 = 10', '10 + 10 = 20']

=== 세 번째 계산 ===
계산: 20 + 10 = 30
결과: total=30, history=['0 + 10 = 10', '10 + 10 = 20', '20 + 10 = 30']


<br>

- **메모리 동작 흐름**

```python
1. 첫 번째 실행
   invoke({"total": 0}, config) → 실행 → 결과 저장
                                         ↓
                                   메모리: {"total": 10}

2. 두 번째 실행  
   invoke({}, config) → 메모리에서 이전 상태 로드 → 실행 → 결과 저장
         ↑                      ↓                          ↓
     새 입력 없음        {"total": 10}              메모리: {"total": 20}
```

<br>

#### 사용자별 방문 카운터 예시

In [16]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict

- 상태 정의

In [17]:
class VisitorState(TypedDict):
    name: str           # 방문자 이름
    visit_count: int    # 방문 횟수
    message: str        # 환영 메시지

- 노드 함수

In [18]:
def welcome_visitor(state: VisitorState) -> dict:
    """방문 횟수에 따라 다른 인사를 합니다"""
    name = state["name"]
    count = state["visit_count"] + 1

    # 방문 횟수에 따른 메시지 분기
    if count == 1:
        message = f"환영합니다, {name}님! 처음 오셨네요!"
    elif count < 5:
        message = f"반갑습니다, {name}님! ({count}번째 방문)"
    else:
        message = f"{name}님, 단골이시네요! ({count}번째 방문)"

    return {"visit_count": count, "message": message}

- 그래프 구성

In [19]:
graph = StateGraph(VisitorState)
graph.add_node("welcome", welcome_visitor)
graph.add_edge(START, "welcome")
graph.add_edge("welcome", END)

- 메모리 설정

In [20]:
memory = InMemorySaver()
app = graph.compile(checkpointer=memory)

- 실행

In [21]:
# Alice 설정 (thread_id로 구분)
alice_config = {"configurable": {"thread_id": "user_alice"}}

# Bob 설정
bob_config = {"configurable": {"thread_id": "user_bob"}}

In [22]:
print("=" * 50)
print("Alice 첫 방문")
result = app.invoke({"name": "Alice", "visit_count": 0, "message": ""}, config=alice_config)
print(result["message"])

print("\n" + "=" * 50)
print("Bob 첫 방문")
result = app.invoke({"name": "Bob", "visit_count": 0, "message": ""}, config=bob_config)
print(result["message"])

print("\n" + "=" * 50)
print("Alice 두 번째 방문")
# name만 전달 - visit_count는 메모리에서 자동 복원!
result = app.invoke({"name": "Alice", "message": ""}, config=alice_config)
print(result["message"])

print("\n" + "=" * 50)
print("Alice 세 번째 ~ 다섯 번째 방문")
for i in range(3):
    result = app.invoke({"name": "Alice", "message": ""}, config=alice_config)
    print(result["message"])

Alice 첫 방문
환영합니다, Alice님! 처음 오셨네요!

Bob 첫 방문
환영합니다, Bob님! 처음 오셨네요!

Alice 두 번째 방문
반갑습니다, Alice님! (2번째 방문)

Alice 세 번째 ~ 다섯 번째 방문
반갑습니다, Alice님! (3번째 방문)
반갑습니다, Alice님! (4번째 방문)
Alice님, 단골이시네요! (5번째 방문)


<br>

```python
Alice (thread_id: "user_alice")     Bob (thread_id: "user_bob")
┌─────────────────────────┐        ┌─────────────────────────┐
│ visit_count: 5          │        │ visit_count: 1          │
│ name: "Alice"           │        │ name: "Bob"             │
└─────────────────────────┘        └─────────────────────────┘
        ↑                                  ↑
        │                                  │
    별개의 메모리 공간!              별개의 메모리 공간!
```

<br>

### 대화 기록 저장 챗봇 예시


In [23]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Annotated
from datetime import datetime

- 상태 정의

In [24]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]  # ← reducer로 자동 누적!
    user_input: str

- 노드 함수

In [25]:
def process_message(state: ChatState) -> dict:
    """사용자 메시지를 받아 응답을 생성합니다"""
    user_input = state["user_input"]
    messages = state.get("messages", [])  # 기존 메시지 읽기

    # 사용자 메시지
    user_msg = {
        "role": "user",
        "content": user_input,
        "time": datetime.now().strftime("%H:%M")
    }

    # 응답 생성
    if "안녕" in user_input:
        response = "안녕하세요! 무엇을 도와드릴까요?"
    elif "이름" in user_input:
        response = "저는 LangGraph 챗봇입니다!"
    elif "기억" in user_input or "대화" in user_input:
        # 기존 메시지에서 user 메시지 수 + 현재 메시지
        history_count = len([m for m in messages if getattr(m, 'type', None) == 'human' or (isinstance(m, dict) and m.get("role") == "user")]) + 1
        response = f"네, 지금까지 {history_count}번의 대화를 나눴습니다!"
    else:
        response = f"'{user_input}'에 대해 말씀해 주셨네요."

    assistant_msg = {
        "role": "assistant", 
        "content": response,
        "time": datetime.now().strftime("%H:%M")
    }

    # add_messages reducer가 자동으로 누적시킴
    return {"messages": [user_msg, assistant_msg]}

- 그래프 구성 및 메모리 연결

In [27]:
graph = StateGraph(ChatState)
graph.add_node("chat", process_message)
graph.add_edge(START, "chat")
graph.add_edge("chat", END)

memory = InMemorySaver()
app = graph.compile(checkpointer=memory)

- 실행

In [28]:
session_config = {"configurable": {"thread_id": "chat_session_001"}}

In [29]:
def chat(user_message: str) -> str:
    """사용자 메시지를 보내고 응답을 받습니다"""
    result = app.invoke(
        {"user_input": user_message}, 
        config=session_config
    )
    return result["messages"][-1].content

In [31]:
print("🤖 챗봇: 대화를 시작합니다!\n")

print(f"👤 사용자: 안녕!")
print(f"🤖 챗봇: {chat('안녕!')}\n")

print(f"👤 사용자: 네 이름이 뭐야?")
print(f"🤖 챗봇: {chat('네 이름이 뭐야?')}\n")

print(f"👤 사용자: 오늘 날씨 좋다")
print(f"🤖 챗봇: {chat('오늘 날씨 좋다')}\n")

print(f"👤 사용자: 우리 몇 번 대화했지?")
print(f"🤖 챗봇: {chat('우리 몇 번 대화했지?')}\n")

# # 전체 대화 기록 확인
# print("=" * 50)
# print("📝 전체 대화 기록:")
# state = app.get_state(session_config)
# for msg in state.values["messages"]:
#     role = "👤" if msg.type == "human" else "🤖"
#     print(f"  {role} {msg.content}")

🤖 챗봇: 대화를 시작합니다!

👤 사용자: 안녕!
🤖 챗봇: 안녕하세요! 무엇을 도와드릴까요?

👤 사용자: 네 이름이 뭐야?
🤖 챗봇: 저는 LangGraph 챗봇입니다!

👤 사용자: 오늘 날씨 좋다
🤖 챗봇: '오늘 날씨 좋다'에 대해 말씀해 주셨네요.

👤 사용자: 우리 몇 번 대화했지?
🤖 챗봇: 네, 지금까지 8번의 대화를 나눴습니다!



<br>

#### 메모리 상태 확인 : `get_state()`

- **현재 상태 가져오기**

In [ ]:
state = app.get_state(config)

print(state.values)          # 실제 데이터
print(state.config)          # 설정 정보
print(state.created_at)      # 생성 시간
print(state.parent_config)   # 이전 상태 참조 

<br>

#### `create_agent`로 메모리 사용
- `LangChain` v1.0의 `create_agent`를 사용하면, 메모리가 내장된 에이전트를 간결하게 만들 수 있음

In [32]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

- llm 초기화

In [33]:
llm = init_chat_model("openai:gpt-4.1-mini")

- 에이전트 생성

In [34]:
agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=InMemorySaver(),
    system_prompt="당신은 친절한 한국어 챗봇입니다.",
)

- 실행 - `thread_id`로 대화 세션 구분

In [35]:
config = {"configurable": {"thread_id": "user_123"}}

In [36]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "안녕! 내 이름은 철수야"}]},
    config=config,
)
print(response["messages"][-1].content)

안녕, 철수야! 만나서 반가워. 오늘 기분은 어때?


In [37]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐였지?"}]},
    config=config,
)
print(response["messages"][-1].content) 

네 이름은 철수야. 기억하고 있어! 무엇을 도와줄까?


<br>

- `StateGraph` 방식과의 비교

<table>
<thead>
<tr>
<th style="text-align: left;">항목</th>
<th style="text-align: left;">StateGraph</th>
<th style="text-align: left;">create_agent</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">상태 정의</td>
<td style="text-align: left;"><code>TypedDict</code> 직접 정의</td>
<td style="text-align: left;">자동 (<code>MessagesState</code>)</td>
</tr>
<tr>
<td style="text-align: left;">노드/엣지</td>
<td style="text-align: left;">수동 구성</td>
<td style="text-align: left;">자동 구성</td>
</tr>
<tr>
<td style="text-align: left;">메모리</td>
<td style="text-align: left;"><code>compile(checkpointer=...)</code></td>
<td style="text-align: left;"><code>create_agent(checkpointer=...)</code></td>
</tr>
<tr>
<td style="text-align: left;">유연성</td>
<td style="text-align: left;">높음 (완전 커스텀)</td>
<td style="text-align: left;">중간 (표준 에이전트)</td>
</tr>
<tr>
<td style="text-align: left;">코드량</td>
<td style="text-align: left;">많음</td>
<td style="text-align: left;">적음</td>
</tr>
</tbody>
</table>

<br>

### 체크포인터 종류

<br>

#### 체크포인터 비교

<table>
<thead>
<tr>
<th style="text-align: left;">체크포인터</th>
<th style="text-align: center;">영속성</th>
<th style="text-align: center;">성능</th>
<th style="text-align: center;">설정</th>
<th style="text-align: left;">권장 환경</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><strong>InMemorySaver</strong></td>
<td style="text-align: center;">❌ (휘발성)</td>
<td style="text-align: center;">최고</td>
<td style="text-align: center;">간단</td>
<td style="text-align: left;">개발, 테스트</td>
</tr>
<tr>
<td style="text-align: left;"><strong>SqliteSaver</strong></td>
<td style="text-align: center;">✅ (파일)</td>
<td style="text-align: center;">좋음</td>
<td style="text-align: center;">간단</td>
<td style="text-align: left;">로컬 개발, 소규모</td>
</tr>
<tr>
<td style="text-align: left;"><strong>PostgresSaver</strong></td>
<td style="text-align: center;">✅ (DB)</td>
<td style="text-align: center;">좋음</td>
<td style="text-align: center;">복잡</td>
<td style="text-align: left;">프로덕션</td>
</tr>
</tbody>
</table>

<br>

#### 체크포인터 선택

<table>
<thead>
<tr>
<th style="text-align: left;">환경</th>
<th style="text-align: left;">체크포인터</th>
<th style="text-align: left;">이유</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><strong>로컬 개발</strong></td>
<td style="text-align: left;">InMemorySaver</td>
<td style="text-align: left;">빠른 설정, 빠른 실행</td>
</tr>
<tr>
<td style="text-align: left;"><strong>테스트/CI</strong></td>
<td style="text-align: left;">InMemorySaver</td>
<td style="text-align: left;">격리된 테스트, 빠른 실행</td>
</tr>
<tr>
<td style="text-align: left;"><strong>프로토타입</strong></td>
<td style="text-align: left;">SqliteSaver</td>
<td style="text-align: left;">영속성 필요, 간단한 설정</td>
</tr>
<tr>
<td style="text-align: left;"><strong>단일 서버 운영</strong></td>
<td style="text-align: left;">SqliteSaver</td>
<td style="text-align: left;">파일 기반 백업, 간편한 관리</td>
</tr>
<tr>
<td style="text-align: left;"><strong>다중 서버 운영</strong></td>
<td style="text-align: left;">PostgresSaver</td>
<td style="text-align: left;">공유 상태, 고가용성</td>
</tr>
<tr>
<td style="text-align: left;"><strong>엔터프라이즈</strong></td>
<td style="text-align: left;">PostgresSaver</td>
<td style="text-align: left;">트랜잭션 보장, 확장성</td>
</tr>
</tbody>
</table>

<br>

### `InMemorySaver`
- `LangGraph`에서 제공하는 가장 간단한 체크포인터
- 별도의 데이터베이스 설정이나 외부 의존성 없이 메모리 내에서 상태를 저장 $\rightarrow$ 개발 및 테스트 단계에서 사용

<table>
<thead>
<tr>
<th style="text-align: left;">특징</th>
<th style="text-align: left;">설명</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><strong>영속성</strong></td>
<td style="text-align: left;">휘발성 (프로세스 종료 시 소멸)</td>
</tr>
<tr>
<td style="text-align: left;"><strong>성능</strong></td>
<td style="text-align: left;">최고 (메모리 기반)</td>
</tr>
<tr>
<td style="text-align: left;"><strong>설정 복잡도</strong></td>
<td style="text-align: left;">매우 간단 (import만으로 사용 가능)</td>
</tr>
<tr>
<td style="text-align: left;"><strong>권장 환경</strong></td>
<td style="text-align: left;">개발, 테스트, 프로토타입, 데모</td>
</tr>
<tr>
<td style="text-align: left;"><strong>다중 스레드</strong></td>
<td style="text-align: left;">지원 (thread_id로 구분)</td>
</tr>
</tbody>
</table>

<br>

#### 챗봇 예시
- `InMemorySaver`를 사용한 예시

In [40]:
from typing import TypedDict
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

- 모델 초기화

In [41]:
model = init_chat_model("openai:gpt-4.1-mini")

- 노드 정의

In [42]:
def chatbot_node(state: MessagesState) -> dict:
    """
    사용자 메시지를 받아 AI 응답을 생성하는 노드

    Args:
        state: MessagesState - 대화 히스토리가 포함된 상태

    Returns:
        dict - AI 응답 메시지를 포함한 딕셔너리
    """
    # 모델을 호출하여 응답 생성
    response = model.invoke(state["messages"])

    # 응답을 messages 키에 담아 반환
    return {"messages": [response]}

- 체크포인터 생성

In [43]:
checkpointer = InMemorySaver()

- 그래프 구성

In [44]:
builder = StateGraph(MessagesState)

# 챗봇 노드 추가
builder.add_node("chatbot", chatbot_node)

# 엣지 연결: START -> chatbot -> END
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

- 컴파일 및 체크포인트 연결

In [45]:
graph = builder.compile(checkpointer=checkpointer)

- 실행

In [ ]:
config = {"configurable": {"thread_id": "conversation_1"}}

In [ ]:
result1 = graph.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요! 제 이름은 철수입니다."}]},
    config
)
print("AI 응답 1:", result1["messages"][-1].content)

AI 응답 1: 안녕하세요, 철수님! 만나서 반갑습니다. 어떻게 도와드릴까요?


In [47]:
result2 = graph.invoke(
    {"messages": [{"role": "user", "content": "제 이름이 뭐였죠?"}]},
    config
)
print("AI 응답 2:", result2["messages"][-1].content)

AI 응답 2: 철수님이라고 하셨습니다! 어떻게 도와드릴까요?


- **상태 검사**
  - 체크포인터에 저장된 상태를 확인

In [48]:
from pprint import pprint

In [ ]:
# 특정 thread의 현재 상태 가져오기
state = graph.get_state(config)

# 저장된 값 출력
print("=== 저장된 상태 값 ===")
pprint(state.values)

# 설정 정보 출력
print("\n=== 설정 정보 ===")
pprint(state.config)

# 메시지 개수 확인
print(f"\n총 메시지 수: {len(state.values['messages'])}")

=== 저장된 상태 값 ===
{'messages': [HumanMessage(content='안녕하세요! 제 이름은 철수입니다.', additional_kwargs={}, response_metadata={}, id='48ac95b4-c3e6-4315-a043-790129448b16'),
              AIMessage(content='안녕하세요, 철수님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 17, 'total_tokens': 37, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_cc588ed157', 'id': 'chatcmpl-Dk3mhdpEx4OJ35FbIDM08SqhMjSpg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e6884-fb30-77a2-8736-31b94551c586-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 20, 'total_tokens': 37, 'input_token_details': {'aud

<br>

#### 다중 사용자 세션 관리
- **`InMemorySaver`는 `thread_id`를 사용하여 여러 사용자의 독립적인 대화 세션을 동시에 관리 가능**

In [50]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain.chat_models import init_chat_model

- llm 정의

In [53]:
model = init_chat_model("openai:gpt-4.1-mini")

- 챗봇 노드 함수

In [54]:
def chatbot_node(state: MessagesState) -> dict:
    response = model.invoke(state["messages"])
    return {"messages": [response]}

- 그래프 구성

In [55]:
builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

- 체크포인터 연결 및 그래프 컴파일

In [56]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

- 각 사용자별 독립된 `config` 생성

In [57]:
user1_config = {"configurable": {"thread_id": "user_alice"}}
user2_config = {"configurable": {"thread_id": "user_bob"}}

- 실행

In [59]:
# 사용자 1(Alice)의 대화
graph.invoke(
    {"messages": [{"role": "user", "content": "제 이름은 Alice입니다."}]},
    user1_config
)

{'messages': [HumanMessage(content='제 이름은 Alice입니다.', additional_kwargs={}, response_metadata={}, id='b23e9297-e6fa-404a-a008-c7ca7c524b0f'),
  AIMessage(content='안녕하세요, Alice님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 13, 'total_tokens': 32, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_59b5bbb72f', 'id': 'chatcmpl-Dk3pEU8GJhEveOPuoIVqKK58ZpvOJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e6887-652a-7472-a1d7-8570dad51fa8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 19, 'total_tokens': 32, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'out

In [60]:
# 사용자 2(Bob)의 대화
graph.invoke(
    {"messages": [{"role": "user", "content": "제 이름은 Bob입니다."}]},
    user2_config
)

{'messages': [HumanMessage(content='제 이름은 Bob입니다.', additional_kwargs={}, response_metadata={}, id='ea1ba395-258c-4bbc-9904-2a0df47586c9'),
  AIMessage(content='만나서 반갑습니다, Bob님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 13, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_67958e4f8a', 'id': 'chatcmpl-Dk3pJQWlOnRmd3fwbfpSBfvROUL6F', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e6887-68f9-7b00-a7ed-d842bdf9b061-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 17, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_d

In [61]:
# 각 사용자의 이름 기억 확인
result_alice = graph.invoke(
    {"messages": [{"role": "user", "content": "제 이름이 뭐죠?"}]},
    user1_config
)
print("Alice에게 응답:", result_alice["messages"][-1].content)

result_bob = graph.invoke(
    {"messages": [{"role": "user", "content": "제 이름이 뭐죠?"}]},
    user2_config
)

print("Bob에게 응답:", result_bob["messages"][-1].content)

Alice에게 응답: 고객님께서 말씀해주신 이름은 Alice입니다. 도움이 필요하신 것이 있을까요?
Bob에게 응답: 당신의 이름은 Bob입니다. 어떻게 도와드릴까요?


- 각 사용자의 상태 독립적으로 조회

In [62]:
alice_state = graph.get_state(user1_config)
bob_state = graph.get_state(user2_config)

In [63]:
print(f"\nAlice 메시지 수: {len(alice_state.values['messages'])}")
print(f"Bob 메시지 수: {len(bob_state.values['messages'])}")


Alice 메시지 수: 6
Bob 메시지 수: 6


<br>

#### `InMemoryStore` : Semantic 검색을 지원하는 장기 메모리
- **`InMemorySaver`가 단기 메모리 (`thread`내 대화)를 담당한다면, `InMemoryStore`는 `thread`를 넘나드는 장기 메모리를 지원**
- `index` 파라미터를 사용하며누 임베딩 기반 시맨틱 검색도 가능

<br>

- **`index` 파라미터로 시맨틱 검색 활성화**
  

In [69]:
from langgraph.store.memory import InMemoryStore
from langchain.embeddings import init_embeddings

- 임베딩 모델 초기화

In [70]:
embeddings = init_embeddings("openai:text-embedding-3-small")

- `index` 파라미터로 시맨틱 검색 활성화

In [71]:
store = InMemoryStore(
    index={
        "embed": embeddings,   # 임베딩 모델
        "dims": 1536,          # 임베딩 차원 (text-embedding-3-small: 1536)
        "fields": ["text"],    # 임베딩할 필드명
    }
)

- `Namespace` 기반 저장

In [72]:
store.put(("user_123", "preferences"), "food", {"text": "매운 음식을 좋아합니다"})
store.put(("user_123", "preferences"), "hobby", {"text": "등산과 독서를 즐깁니다"})
store.put(("user_123", "preferences"), "music", {"text": "클래식 음악을 즐겨 듣습니다"})

- **시맨틱 검색 - 의미적으로 유사한 항목 검색**

In [74]:
results = store.search(("user_123", "preferences"), query="매운 음식 취향", limit=2)
for item in results:
    print(f"- {item.value['text']} (score: {item.score:.3f})")

- 매운 음식을 좋아합니다 (score: 0.766)
- 클래식 음악을 즐겨 듣습니다 (score: 0.333)


- **`index` 없이 사용 (키-값 검색만)**


In [75]:
store_simple = InMemoryStore()

In [ ]:
store_simple.put(("user_123", "profile"), "name", {"value": "김철수"})

# 키로 직접 조회
item = store_simple.get(("user_123", "profile"), "name")
print(item.value)

{'value': '김철수'}


<br>

#### `InMemorySaver` vs `InMemoryStore`

<table>
<thead>
<tr>
<th style="text-align: left;">구분</th>
<th style="text-align: left;">InMemorySaver</th>
<th style="text-align: left;">InMemoryStore</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><strong>역할</strong></td>
<td style="text-align: left;">단기 메모리 (thread-scoped)</td>
<td style="text-align: left;">장기 메모리 (cross-thread)</td>
</tr>
<tr>
<td style="text-align: left;"><strong>범위</strong></td>
<td style="text-align: left;">단일 대화 세션</td>
<td style="text-align: left;">사용자/앱 전체</td>
</tr>
<tr>
<td style="text-align: left;"><strong>검색</strong></td>
<td style="text-align: left;">checkpoint_id로 조회</td>
<td style="text-align: left;">key-value 또는 시맨틱 검색</td>
</tr>
<tr>
<td style="text-align: left;"><strong>시맨틱 검색</strong></td>
<td style="text-align: left;">미지원</td>
<td style="text-align: left;">index 파라미터로 지원</td>
</tr>
<tr>
<td style="text-align: left;"><strong>영속성</strong></td>
<td style="text-align: left;">휘발성</td>
<td style="text-align: left;">휘발성 (프로세스 종료 시 소멸)</td>
</tr>
</tbody>
</table>

<br>

<hr>